# Agent Tribunal — Dev Log

## Objetivo

Fecha um TODO real deixado pelo `policy_engine` no V1: quando várias
`PolicyDecision` se aplicam ao mesmo cenário, quem decide o veredito final?
`adjudicate()` aplica uma regra determinística — a decisão mais restritiva
vence (`DENY` > `REQUIRES_HUMAN_REVIEW` > `ALLOW_WITH_MITIGATION` >
`ALLOW`).

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from core.agent_tribunal.tribunal import adjudicate
from core.policy_engine.engine import evaluate
from shared.schemas import DataCategory, LegalBasis

decisions = evaluate(data_categories=[DataCategory.SENSITIVE], legal_basis=LegalBasis.NOT_DETERMINED, context={"data_subtype": "health"})
print(f"{len(decisions)} decisão(ões) real(is) do policy_engine para dado de saúde sem base legal:")
for d in decisions:
    print(f"  - {d.policy_id}: {d.status.value} ({d.risk_level.value})")
verdict = adjudicate(decisions)
print()
print(f"Veredito do tribunal: {verdict.final_status.value} (risco={verdict.risk_level.value})")
print(verdict.rationale)

2 decisão(ões) real(is) do policy_engine para dado de saúde sem base legal:
  - POL-001: requires_human_review (critical)
  - POL-008: requires_human_review (medium)

Veredito do tribunal: requires_human_review (risco=critical)
2 decisões concorrentes consideradas (POL-001, POL-008). Vencedora (mais restritiva): POL-001 — REQUER REVISÃO HUMANA. Dado de saúde sem consentimento explícito registrado e/ou sem RIPD concluído. Art. 11, I e Art. 38 da LGPD exigem consentimento específico e destacado e RIPD para tratamento de dado sensível em larga escala — revisão humana obrigatória antes de prosseguir. Demais decisões consideradas mas superadas por precedência: POL-008.


## Testes e Handoff

```
"C:/Users/Yuri_/.venvs/athenagov-ai/Scripts/python.exe" -m pytest core/agent_tribunal/tests -v
```

7/7 testes passando, incluindo adjudicação de decisões reais do
`policy_engine` (não só fixtures sintéticas).